In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch

In [2]:
housing = fetch_california_housing()

In [3]:

X_train, X_test, y_train, y_test = train_test_split(housing.data, housing.target, random_state=42)

In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, random_state=42)

normalize the data, using tensor operations instead of `StandardScaler`

In [5]:

X_train = torch.FloatTensor(X_train) # FloatTensor turns the 64bit numpy array X_train to a 32bit tensor (instead of a 62 bit)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)

means = X_train.mean(dim=0, keepdims=True) # average of the columns, collapsing the row. gives us the mean per feature
stds = X_train.std(dim=0, keepdims=True) # same


# normalizing step
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

we also need to convert our targets to tensors

since our preds will be column vectors, we need to ensure our targets are also columns vectors

originally its a 1d array of prices, so we need to reshape to column vectors (-1 is a wildcard for rows, 1 col per row)

In [6]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_valid).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [7]:
# parameters (rand weights instead of 0 to break symmetry)
torch.manual_seed(42)
n_features = X_train.shape[1] # 8 input features
w = torch.randn((n_features, 1), requires_grad=True) # weights of each feature, 8x1 tensor of random values (column vector)
b = torch.tensor(0., requires_grad=True) # bias param

this is a single layer nn, so 1 set of weights for the inputs, and 1 output neuron  matmul the input features w the weights and add bias to get our output, then get loss and perform gradient descent (batch gd in this case)

In [8]:
learning_rate = 0.4
n_epochs = 100

for epoch in range(n_epochs):
    y_pred = X_train @ w + b  # (whatever, 1) vector of predictions matching w/ the size of y_train
    loss = ((y_pred - y_train) ** 2).mean() # MSE
    loss.backward() # gradients of the loss w/ regards to each of the 8 input features and the bias 
    with torch.no_grad():
        print(b.grad)
        print(w.grad)
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad # w.grad is 8,1 matrix
        b.grad.zero_()
        w.grad.zero_()
    print(f'Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}')
    

tensor(-4.1643)
tensor([[-1.1391],
        [ 0.6704],
        [ 1.3922],
        [ 1.3897],
        [-2.9650],
        [-0.6495],
        [ 6.2128],
        [-5.5244]])
Epoch 1/100, Loss: 16.158458709716797
tensor(-0.8328)
tensor([[-0.1458],
        [-0.9254],
        [-1.1661],
        [-1.1926],
        [ 0.7602],
        [ 0.5226],
        [-3.3911],
        [ 3.7948]])
Epoch 2/100, Loss: 4.87936544418335
tensor(-0.1666)
tensor([[-0.0317],
        [ 0.0908],
        [ 0.9437],
        [ 0.7035],
        [-0.8431],
        [-0.2203],
        [ 2.3798],
        [-1.9177]])
Epoch 3/100, Loss: 2.255225896835327
tensor(-0.0333)
tensor([[-0.0845],
        [-0.1974],
        [-0.5820],
        [-0.6768],
        [ 0.3245],
        [ 0.1860],
        [-1.1485],
        [ 1.4693]])
Epoch 4/100, Loss: 1.3307628631591797
tensor(-0.0067)
tensor([[ 0.0289],
        [ 0.0403],
        [ 0.5006],
        [ 0.3197],
        [-0.2869],
        [-0.0921],
        [ 0.9783],
        [-0.6102]])
Epoch 

In [9]:
X_new = X_test[:3] # pretend like this is new

with torch.no_grad(): # doenst keep track of computation graph so faster
    y_pred = X_new @ w + b # use trained param to make preds

print(y_pred)
print(y_test[:3])
print(y_pred - y_test[:3])

tensor([[0.7090],
        [1.7824],
        [2.7693]])
tensor([[0.4770],
        [0.4580],
        [5.0000]])
tensor([[ 0.2320],
        [ 1.3244],
        [-2.2307]])


## Rewrite using PyTorch's High Level API 

In [10]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1) # 8 inputs, 1 neuron (output neuron)

In [11]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [12]:
model.weight # only 1 row, since 1 neuron, 1 row per neuron

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [13]:
for param in model.parameters(): # weights and bias
    print(param)

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)
Parameter containing:
tensor([0.3117], requires_grad=True)


In [14]:
model(X_train[:2]) # not trained yet, preds terrible
# model internall calls forward() method (X_train[:2] @ self.weight.T + self.bias)

tensor([[-0.4718],
        [ 0.1131]], grad_fn=<AddmmBackward0>)

create an optimizer to update model params, and choose a loss function
- optimizer: stochastic gradient descent (can be used for sgd, mini batch gd, or batch gd)
- loss function: mean squared error (mse)

optimizer is the object that actually updates the model params using the gradients that autograd created
- give it model.parameters(), an iterator over all parameters in the model so it knows what to update
- learning rate, for how much to update based on the .grad of each param

In [15]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  
mse = nn.MSELoss()

In [16]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train) # call, loss function (i.e. mse in this case), `loss` is a tensor w/ .grad_fn
        loss.backward() # compute gradients and store them in each params .grad. 
                        # loss has computation graph, with y_pred, so it can walk backwards to orignal w and b
        optimizer.step() # take the gradients and decide how to apply them (update w and b)
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [17]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/100, Loss: 4.3378496170043945
Epoch 2/100, Loss: 0.7802932858467102
Epoch 3/100, Loss: 0.6253840923309326
Epoch 4/100, Loss: 0.6060433983802795
Epoch 5/100, Loss: 0.5956298112869263
Epoch 6/100, Loss: 0.5873566269874573
Epoch 7/100, Loss: 0.5802990198135376
Epoch 8/100, Loss: 0.5741382241249084
Epoch 9/100, Loss: 0.5687100291252136
Epoch 10/100, Loss: 0.5639079213142395
Epoch 11/100, Loss: 0.5596511363983154
Epoch 12/100, Loss: 0.555873692035675
Epoch 13/100, Loss: 0.5525194406509399
Epoch 14/100, Loss: 0.5495391488075256
Epoch 15/100, Loss: 0.5468899011611938
Epoch 16/100, Loss: 0.544533908367157
Epoch 17/100, Loss: 0.5424376726150513
Epoch 18/100, Loss: 0.5405715703964233
Epoch 19/100, Loss: 0.5389097332954407
Epoch 20/100, Loss: 0.5374287962913513
Epoch 21/100, Loss: 0.536108672618866
Epoch 22/100, Loss: 0.5349311232566833
Epoch 23/100, Loss: 0.5338802933692932
Epoch 24/100, Loss: 0.5329418182373047
Epoch 25/100, Loss: 0.532103419303894
Epoch 26/100, Loss: 0.5313538908958435

In [18]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = model(X_new)

y_pred # similar to prev, but not same b/c nn.Linear init parms slightly diff 

tensor([[0.7070],
        [1.7841],
        [2.7667]])

## Implementing a Regression MLP

`nn.Sequential` module chains multiple modules
- when we call this module with some inputs, it feeds the inputs into the first model
- then feeds the outputs of the first model into the second model, etc. etc. 

MLP is just a stack of modules, two hidden layers and one output layer (of perceptrons)

So we can use `nn.Sequential` to build a MLP for our task, w/ multiple chained linear functions instead of just one like before.

In [19]:
torch.manual_seed(42)

# an multilayer perceptron is just multiple layers of perceptrons where: 
# each perceptron is a linear function matmul the weights and adds the bias
# then applies a nonlinear activation function to its output
# and each feeds its output to each of the perceptrons in the outer layer

# in this case hidden layer 1 has 50 neurons giving us 50 inputs for the neurons in layer 2, 
#   hidden layer 2 has 40, and 1 output neuron
# 8 -> 50 -> 40 -> 1
model = nn.Sequential(
    nn.Linear(n_features, 50), # perceptron, 8 input features, 50 outputs (50 perceptrons)
    nn.ReLU(), # activation function, acts itemwise so shape of output = shape of input (50)
    nn.Linear(50, 40), # 50 inputs (same as output of prev layer), 40 outputs
    nn.ReLU(),
    nn.Linear(40, 1) # 40 inputs, 1 output
)

In [20]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/100, Loss: 5.045480251312256
Epoch 2/100, Loss: 2.0523128509521484
Epoch 3/100, Loss: 1.0039883852005005
Epoch 4/100, Loss: 0.8570139408111572
Epoch 5/100, Loss: 0.7740675210952759
Epoch 6/100, Loss: 0.7225848436355591
Epoch 7/100, Loss: 0.6893726587295532
Epoch 8/100, Loss: 0.6669033169746399
Epoch 9/100, Loss: 0.6507739424705505
Epoch 10/100, Loss: 0.6383934617042542
Epoch 11/100, Loss: 0.6281994581222534
Epoch 12/100, Loss: 0.6193398833274841
Epoch 13/100, Loss: 0.6113173365592957
Epoch 14/100, Loss: 0.6038705110549927
Epoch 15/100, Loss: 0.5968308448791504
Epoch 16/100, Loss: 0.5901119112968445
Epoch 17/100, Loss: 0.583646833896637
Epoch 18/100, Loss: 0.5774063467979431
Epoch 19/100, Loss: 0.5713555216789246
Epoch 20/100, Loss: 0.565444827079773
Epoch 21/100, Loss: 0.5596968531608582
Epoch 22/100, Loss: 0.5540881752967834
Epoch 23/100, Loss: 0.5486184358596802
Epoch 24/100, Loss: 0.5432835817337036
Epoch 25/100, Loss: 0.5380954146385193
Epoch 26/100, Loss: 0.533028244972229

## Implementing Mini-Batch GD Using DataLoaders

`DataLoader` can load batches of data of a desired size, and shuffle the data at each epoch if we want. 

But it needs `__len__(self)` to get the # of samples in the dataset, and `__getitem__(self, index)` to load the sample at the given index *including the target*, so we need one object instead of two tensors, so the loader can get a tuple of the training item and the target

So we need to turn our training set from *two Tensors* to *one DataSet* object, by wrapping them. 

We also want our dataset to be shuffled, so the batches have different instances for each epoch

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train) # wrap tensors in a dataset object
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) 

create a new untrained instance of the model, and a loss function and optimizer (w/ a lower learning rate)

In [22]:
# same model as last time, but reinit / untrained
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(n_features, 50), # perceptron, 8 input features, 50 outputs (50 perceptrons)
    nn.ReLU(), # activation function, acts itemwise so shape of output = shape of input (50)
    nn.Linear(50, 40), # 50 inputs (same as output of prev layer), 40 outputs
    nn.ReLU(),
    nn.Linear(40, 1) # 40 inputs, 1 output
)

In [23]:
device = 'mps'
model = model.to(device) # move the model to our gpu

In [24]:
learning_rate = 0.02
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [25]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train() # switches model and submodules to training mode
    for epoch in range(n_epochs): 
        total_loss = 0. # total loss over the whole epoch
        for X_batch, y_batch in train_loader: # iterate through training set, 1 batch (of size 32) at a time 
            X_batch, y_batch = X_batch.to(device), y_batch.to(device) # copy batches to gpu
            y_pred = model(X_batch) # make preds, forward pass
            loss = criterion(y_pred, y_batch) # compute loss (mse) for this batch (loss is a tensor w/ computation graph)
            total_loss += loss.item() # update total loss for epoch by value of this batch's loss
            loss.backward() # compute gradients
            optimizer.step() # update weights and biases
            optimizer.zero_grad() # clear gradients for next batch / epoch

        mean_loss = total_loss / len(train_loader) # avg loss over all batches so far, len(train_loader) is # of batches
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [26]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/100, Loss: 0.5900
Epoch 2/100, Loss: 0.4046
Epoch 3/100, Loss: 0.3801
Epoch 4/100, Loss: 0.3629
Epoch 5/100, Loss: 0.3529
Epoch 6/100, Loss: 0.3520
Epoch 7/100, Loss: 0.3408
Epoch 8/100, Loss: 0.3427
Epoch 9/100, Loss: 0.3406
Epoch 10/100, Loss: 0.3378
Epoch 11/100, Loss: 0.3304
Epoch 12/100, Loss: 0.3267
Epoch 13/100, Loss: 0.3244
Epoch 14/100, Loss: 0.3221
Epoch 15/100, Loss: 0.3186
Epoch 16/100, Loss: 0.3149
Epoch 17/100, Loss: 0.3123
Epoch 18/100, Loss: 0.3111
Epoch 19/100, Loss: 0.3088
Epoch 20/100, Loss: 0.3072
Epoch 21/100, Loss: 0.3049
Epoch 22/100, Loss: 0.2995
Epoch 23/100, Loss: 0.2993
Epoch 24/100, Loss: 0.2948
Epoch 25/100, Loss: 0.2974
Epoch 26/100, Loss: 0.2945
Epoch 27/100, Loss: 0.2922
Epoch 28/100, Loss: 0.2915
Epoch 29/100, Loss: 0.2913
Epoch 30/100, Loss: 0.2861
Epoch 31/100, Loss: 0.2864
Epoch 32/100, Loss: 0.2855
Epoch 33/100, Loss: 0.2852
Epoch 34/100, Loss: 0.2822
Epoch 35/100, Loss: 0.2809
Epoch 36/100, Loss: 0.2810
Epoch 37/100, Loss: 0.2807
Epoch 38/1

Mini batch was better, because of:
- more weight updates, the weights were updated once per batch instead of once per epoch, so more weight updates / opportunity for improvement
- we also used a diff learning rate
- more noise, so can help the model escape local minima becuase subsets are are a bit noisy

## Model Eval

time to write a func to eval the model. 

it takes the model and a `DataLoader` for the dataset we want to eval on, a function to compute the metric for a given batch (how good the models predictions our, MSE for us, but could be accuracy, MAE, a custom one, whatever), and a function to aggregate the batch metrics (mean default) to get the metric for the whole dataset

In [28]:
def evaluate(model, data_loader, metric_fn, aggregate_fn=torch.mean):
    model.eval() # put model in evaluation mode
    metrics = [] # array of metrics for the whole batch
    with torch.no_grad(): # dont wanna record these calculations in the computation graph
        for X_batch, y_batch in data_loader: # iterate through all the batches (inputs and target)
            X_batch, y_batch = X_batch.to(device), y_batch.to(device) # copy to gpu
            y_pred = model(X_batch) # make preds / forward pass
            metric = metric_fn(y_pred, y_batch) # metric is a scalar tensor i.e tensor(0.41)
            metrics.append(metric) # add it to our array of all the metrics

    # torch.stack turns list of tensors `[tensor(0.41), tensor(0.29), ...]`` into 1 tensor `tensor([0.41, 0.29, ...]`
    return aggregate_fn(torch.stack(metrics)) # so now our fn can operate on it directly

build dataset and dataloader for validation set, and compute validation mse

In [29]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(0.3609, device='mps:0')

suppose we want rmse, pytorch doesnt have built in but we can write ourselves

In [34]:
def rmse(y_pred, y_true):
    return ((y_pred - y_true)**2).mean().sqrt()

evaluate(model, valid_loader, rmse)

tensor(0.5343, device='mps:0')

In [35]:
valid_mse.sqrt() # NOT the same as our RMSE???? (It should be)

tensor(0.6007, device='mps:0')

**ISSUE!!!** Instead of calc rmse of whole validation set, we computed it over each batch and then computed the mean of all these batch rmses. thats not the same as the rmse over the whole validation set. 
- so the rmse is not the same as the sqrt of the mse

solve: use mse as metric, and use the aggregate fn to take the mean mse (correct) and then sqrt it to get rmse (now correct)

In [36]:
evaluate(model, valid_loader, mse, aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(0.6007, device='mps:0')

use TorchMetrics, provides *streaming metrics*
- streaming metric is an object that keeeps track of a given metric, can be updated one batch at a time
- so in it's final state it represents the metric of the whole dataset

In [38]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset() # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration

    return metric.compute()  # compute the final result at the end

In [39]:
 # squared=False returns RMSE, by not squaring it at the end (RMSE = sqrt(MSE))
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_tm(model, valid_loader, rmse)

tensor(0.6008, device='mps:0')

In [44]:
def train(model, optimizer, criterion, train_loader, n_epochs):
    model.train() # switches model and submodules to training mode
    for epoch in range(n_epochs): 
        total_loss = 0. # total loss over the whole epoch
        for X_batch, y_batch in train_loader: # iterate through training set, 1 batch (of size 32) at a time 
            X_batch, y_batch = X_batch.to(device), y_batch.to(device) # copy batches to gpu
            y_pred = model(X_batch) # make preds, forward pass
            loss = criterion(y_pred, y_batch) # compute loss (mse) for this batch (loss is a tensor w/ computation graph)
            total_loss += loss.item() # update total loss for epoch by value of this batch's loss
            loss.backward() # compute gradients
            optimizer.step() # update weights and biases
            optimizer.zero_grad() # clear gradients for next batch / epoch

        mean_loss = total_loss / len(train_loader) # avg loss over all batches so far, len(train_loader) is # of batches
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")
        print(f'Training RMSE:  {evaluate_tm(model, train_loader, rmse)}')
        print(f'Validation RMSE: {evaluate_tm(model, valid_loader, rmse)}')

In [43]:
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/100, Loss: 0.2490
Training RMSE:  0.4934425354003906
Epoch 2/100, Loss: 0.2496
Training RMSE:  0.4947706162929535
Epoch 3/100, Loss: 0.2493
Training RMSE:  0.5057143568992615
Epoch 4/100, Loss: 0.2484
Training RMSE:  0.48975834250450134
Epoch 5/100, Loss: 0.2471
Training RMSE:  0.4943796694278717
Epoch 6/100, Loss: 0.2467
Training RMSE:  0.4947606921195984
Epoch 7/100, Loss: 0.2485
Training RMSE:  0.4913664758205414
Epoch 8/100, Loss: 0.2489
Training RMSE:  0.49269944429397583
Epoch 9/100, Loss: 0.2464
Training RMSE:  0.4924623370170593
Epoch 10/100, Loss: 0.2487
Training RMSE:  0.49299466609954834
Epoch 11/100, Loss: 0.2469
Training RMSE:  0.5047283172607422
Epoch 12/100, Loss: 0.2472
Training RMSE:  0.49455809593200684
Epoch 13/100, Loss: 0.2465
Training RMSE:  0.4929044246673584
Epoch 14/100, Loss: 0.2471
Training RMSE:  0.49574291706085205
Epoch 15/100, Loss: 0.2468
Training RMSE:  0.48623189330101013
Epoch 16/100, Loss: 0.2459
Training RMSE:  0.4972154200077057
Epoch 17/10

## Building Nonsequential Models Using Custom Modules

Nonsequential model means that inputs don't flow straight through all the layers lineraly, they can skip layers, branch, merge, etc. 

Wide and Deep, the nn we are using in this case connects all or part of the inputs directly to the output layer (the short path), and then the traditional deep path, and combines them into the final prediction.

This makes it so that the nn can learn deep patterns via the deep path, and simple patterns via the short path
- for example going through the full network can cause simple rules to become distorted, and they might be fine after just 1 layer in a 10 layer network 

We need to write a custom module for this, since `nn.Sequential` just chains modules together, so we can't do anything more complex 
- create all the layers we need in the ctor after init superclass
- define how layers should be used by the module in the `forward()` method (skip, merge, branch, etc) (skip in this case)

In [50]:
class WideAndDeep(nn.Module): # inherits from nn.Module
    def __init__(self, n_features):
        super().__init__() # initialize superclass (nn.Module)
        self.deep_stack = nn.Sequential( # deep path, traditional sequential net
            nn.Linear(n_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        # seperate output layer, needs to take:
        # 40 outputs from prev hidden layer (deep) and orig feature inputs (short) and give 1 output
        self.output_layer = nn.Linear(40 + n_features, 1) 

        # short path is just features direct to the output layer

    def forward(self, X):
        deep_output = self.deep_stack(X) # deep path outputs
        wide_and_deep = torch.concat([X, deep_output], dim=1) # combine orig features + deep outputs
        return self.output_layer(wide_and_deep) # turn into 1 final output 


In [53]:
torch.manual_seed(42)
model = WideAndDeep(n_features).to(device)
learning_rate = 0.002 # new model, new optimal learning rate
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # new optimizer, updates the new models params
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/100, Loss: 1.4093
Training RMSE:  0.7966051697731018
Validation RMSE: 0.8793566823005676
Epoch 2/100, Loss: 0.6084
Training RMSE:  0.7608559727668762
Validation RMSE: 0.7931070327758789
Epoch 3/100, Loss: 0.5648
Training RMSE:  0.7387374043464661
Validation RMSE: 0.873001217842102
Epoch 4/100, Loss: 0.5270
Training RMSE:  0.7159407138824463
Validation RMSE: 0.7945054173469543
Epoch 5/100, Loss: 0.5048
Training RMSE:  0.6997920870780945
Validation RMSE: 0.7054522633552551
Epoch 6/100, Loss: 0.4810
Training RMSE:  0.6865377426147461
Validation RMSE: 0.6992594599723816
Epoch 7/100, Loss: 0.4642
Training RMSE:  0.6755131483078003
Validation RMSE: 0.6877159476280212
Epoch 8/100, Loss: 0.4501
Training RMSE:  0.6661460995674133
Validation RMSE: 0.6433696746826172
Epoch 9/100, Loss: 0.4396
Training RMSE:  0.6586043834686279
Validation RMSE: 0.6642326712608337
Epoch 10/100, Loss: 0.4301
Training RMSE:  0.6518358588218689
Validation RMSE: 0.6514326930046082
Epoch 11/100, Loss: 0.4222
Tra

In [54]:
evaluate_tm(model, valid_loader, rmse)

tensor(0.5708, device='mps:0')

send a subset of features through the wide path, and a diff subset through the deep path
- split inputs inside forward()

In [61]:
class WideAndDeepV2(nn.Module): # inherits from nn.Module
    def __init__(self, n_features):
        super().__init__() # initialize superclass (nn.Module)
        n_deep_features = n_features - 2
        n_wide_features = 5
        self.deep_stack = nn.Sequential( # deep path, traditional sequential net
            nn.Linear(n_deep_features, 50), nn.ReLU(),
            nn.Linear(50, 40), nn.ReLU(),
        )
        # seperate output layer, needs to take:
        # 40 outputs from prev hidden layer (deep) and orig feature inputs (short) and give 1 output
        self.output_layer = nn.Linear(40 + n_wide_features, 1) 

        # short path is just features direct to the output layer

    def forward(self, X):
        X_wide = X[:, :5]
        X_deep = X[:, 2:]
        deep_output = self.deep_stack(X_deep) # deep path outputs
        wide_and_deep = torch.concat([X_wide, deep_output], dim=1) # combine orig features + deep outputs
        return self.output_layer(wide_and_deep) # turn into 1 final output 

In [62]:
n_epochs = 20
torch.manual_seed(42)
model = WideAndDeepV2(n_features).to(device)
learning_rate = 0.002 
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # new optimizer, updates the new models params
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 1.4730
Training RMSE:  0.8029643893241882
Validation RMSE: 0.9875622987747192
Epoch 2/20, Loss: 0.5910
Training RMSE:  0.7443952560424805
Validation RMSE: 0.7555670142173767
Epoch 3/20, Loss: 0.5329
Training RMSE:  0.7160139083862305
Validation RMSE: 0.7064147591590881
Epoch 4/20, Loss: 0.4996
Training RMSE:  0.6973544955253601
Validation RMSE: 0.6850156188011169
Epoch 5/20, Loss: 0.4776
Training RMSE:  0.6844655871391296
Validation RMSE: 0.6714312434196472
Epoch 6/20, Loss: 0.4630
Training RMSE:  0.6758114099502563
Validation RMSE: 0.6665549874305725
Epoch 7/20, Loss: 0.4532
Training RMSE:  0.66946941614151
Validation RMSE: 0.6627205610275269
Epoch 8/20, Loss: 0.4461
Training RMSE:  0.6650793552398682
Validation RMSE: 0.661758542060852
Epoch 9/20, Loss: 0.4405
Training RMSE:  0.6619570255279541
Validation RMSE: 0.6685302257537842
Epoch 10/20, Loss: 0.4358
Training RMSE:  0.6582766771316528
Validation RMSE: 0.6671285629272461
Epoch 11/20, Loss: 0.4317
Training RMSE:  

In [63]:
evaluate_tm(model, valid_loader, rmse)

tensor(0.6744, device='mps:0')

## Models w/ Multiple Inputs & Outputs (skimmed, not coded)

In [ ]:
# class WideAndDeepV3(nn.Module): # inherits from nn.Module
#     def __init__(self, n_features):
#         super().__init__() # initialize superclass (nn.Module)
#         n_deep_features = n_features - 2
#         n_wide_features = 5
#         self.deep_stack = nn.Sequential( # deep path, traditional sequential net
#             nn.Linear(n_deep_features, 50), nn.ReLU(),
#             nn.Linear(50, 40), nn.ReLU(),
#         )
#         # seperate output layer, needs to take:
#         # 40 outputs from prev hidden layer (deep) and orig feature inputs (short) and give 1 output
#         self.output_layer = nn.Linear(40 + n_wide_features, 1) 

#         # short path is just features direct to the output layer

#     def forward(self, X_wide, X_deep):
#         deep_output = self.deep_stack(X_deep) # deep path outputs
#         wide_and_deep = torch.concat([X_wide, deep_output], dim=1) # combine orig features + deep outputs
#         return self.output_layer(wide_and_deep) # turn into 1 final output 

In [ ]:
# train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
# train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)
# # repeat for validation and test set